# Lab 15-01 · Modeling Dynamic Systems

Four parts, each built on `plants.py`. **Predict first**, then run.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
import plants

## Part 1 · A mechanism and its circuit

*Predict:* the peak displacement of Example 15-01.1 and when it happens.

In [2]:
m, b, k, F0 = 0.5, 4.0, 200.0, 10.0
A, B = plants.msd(m, b, k)
t = np.linspace(0, 3, 3001)
mech = solve_ivp(lambda t, y: A @ y + B * F0, (0, 3), [0, 0], t_eval=t, rtol=1e-9, atol=1e-12)
c = plants.analogs(m, b, k)["force-voltage"]
circ = solve_ivp(lambda t, y: [(F0 - c["R"] * y[0] - y[1] / c["C"]) / c["L"], y[0]], (0, 3), [0, 0], t_eval=t, rtol=1e-9, atol=1e-12)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(t, mech.y[0], label="mass displacement (m)")
ax.plot(t, circ.y[1], "--", label="capacitor charge (C)")
ax.set_xlabel("t (s)")
ax.legend()
k_pk = int(np.argmax(mech.y[0]))
print(f"peak {mech.y[0][k_pk]:.4f} m at {t[k_pk]:.3f} s; largest difference {np.max(np.abs(mech.y[0] - circ.y[1])):.1e}")

peak 0.0763 m at 0.160 s; largest difference 4.2e-17


## Part 2 · The motor's two time constants

In [3]:
R, L, K, J, Bm, V = 2.0, 1e-3, 0.05, 2e-5, 1e-5, 12.0
A2, B2 = plants.motor(R, L, K, J, Bm)
gain, tau = plants.motor_first_order(R, K, J, Bm)
for tt in (0.001, 0.005, 0.05):
    full = plants.step(A2, B2, V, tt)[1]
    first = gain * V * (1 - math.exp(-tt / tau))
    print(f"t = {tt * 1e3:4.0f} ms: full {full:7.2f} rad/s, first-order {first:7.2f} rad/s")
print("time constants (ms):", sorted(-1e3 / np.linalg.eigvals(A2).real))

t =    1 ms: full    8.44 rad/s, first-order   14.54 rad/s
t =    5 ms: full   60.17 rad/s, first-order   64.34 rad/s
t =   50 ms: full  228.59 rad/s, first-order  227.89 rad/s
time constants (ms): [np.float64(0.5166898358913748), np.float64(15.360294291092753)]


## Part 3 · Heating a processor

*Predict:* $T_J$ at 1 s from a one-node model ($\tau$ = 107 s) and from the two-node model.

In [4]:
A3, B3 = plants.thermal_two_node(0.15, 0.35, 5.0, 300.0)
ts = np.logspace(-2, 3, 300)
TJ = [30 + plants.step(A3, B3, 100.0, x)[0] for x in ts]
TC = [30 + plants.step(A3, B3, 100.0, x)[1] for x in ts]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(ts, TJ, label="junction")
ax.semilogx(ts, TC, label="case / heat sink")
ax.set_xlabel("t (s)")
ax.set_ylabel("temperature (°C)")
ax.legend()
print("two-node T_J(1 s) =", round(30 + plants.step(A3, B3, 100.0, 1.0)[0], 1), "°C")
print("one-node T_J(1 s) =", round(30 + 50 * (1 - math.exp(-1 / 107)), 1), "°C")

two-node T_J(1 s) = 41.1 °C
one-node T_J(1 s) = 30.5 °C


## Part 4 · Leakage runaway

In [5]:
for Rth in (0.5, 0.6, 0.65):
    ev = lambda t, y: y[0] - 150
    ev.terminal = True
    sol = solve_ivp(plants.leakage_rhs(60.0, 5.0, 30.0, Rth, 50.0, 25.0), (0, 3000), [25.0], events=ev, rtol=1e-9, atol=1e-9)
    if sol.t_events[0].size:
        print(f"R = {Rth} K/W: passes 150 °C at t = {sol.t_events[0][0]:.0f} s (runaway)")
    else:
        print(f"R = {Rth} K/W: settles at {sol.y[0, -1]:.2f} °C")

R = 0.5 K/W: settles at 64.25 °C
R = 0.6 K/W: settles at 79.38 °C
R = 0.65 K/W: passes 150 °C at t = 462 s (runaway)


**Discussion.** Answer the questions in the lab README. Then change the leakage scale $T_x$ and find how the critical resistance moves.